# Agents as Tools

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to use `as_tool()` to turn agents into callable tools, build the orchestrator pattern with specialist agents, and use `custom_output_extractor` and `is_enabled` for advanced control.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Basic as_tool Usage

Use `agent.as_tool()` to convert an agent into a callable tool. The orchestrator calls it like a function, gets the result back, and continues.

In [ ]:
from agents import Agent, Runner

translator = Agent(
    name="Translator",
    instructions="You translate text to French. Return only the translation.",
)

orchestrator = Agent(
    name="Content Creator",
    instructions="You create content. Use the translator tool to translate text when asked.",
    tools=[translator.as_tool(
        tool_name="translate_to_french",
        tool_description="Translates text to French",
    )],
)

result = await Runner.run(orchestrator, "Write a greeting and translate it to French.")
print(result.final_output)

## 4. The Orchestrator Pattern

A manager agent delegates to specialist agents and synthesizes their results.

In [ ]:
researcher = Agent(
    name="Researcher",
    instructions="You research topics thoroughly. Provide detailed, factual summaries.",
)

writer = Agent(
    name="Writer",
    instructions="You write polished, engaging prose based on provided information.",
)

critic = Agent(
    name="Critic",
    instructions="You review text for accuracy, clarity, and style. Provide specific feedback.",
)

editor = Agent(
    name="Editor",
    instructions=(
        "You manage the content creation process. "
        "1. Use the researcher to gather information. "
        "2. Use the writer to draft content. "
        "3. Use the critic to review it. "
        "Produce the final polished output."
    ),
    tools=[
        researcher.as_tool(tool_name="research", tool_description="Research a topic in depth"),
        writer.as_tool(tool_name="write", tool_description="Write polished content"),
        critic.as_tool(tool_name="review", tool_description="Review content for quality"),
    ],
)

result = await Runner.run(editor, "Create a short article about the history of the internet.")
print(result.final_output)

## 5. Custom Output Extractor

Use `custom_output_extractor` to transform how a specialist's output is returned to the orchestrator.

In [ ]:
async def extract_with_metadata(run_result):
    return f"[Source: {run_result.last_agent.name}] {run_result.final_output}"


researcher_v2 = Agent(
    name="Researcher",
    instructions="You research topics and provide detailed summaries.",
)

manager = Agent(
    name="Manager",
    instructions="You manage research tasks. Use the research tool.",
    tools=[researcher_v2.as_tool(
        tool_name="research",
        tool_description="Research a topic",
        custom_output_extractor=extract_with_metadata,
    )],
)

result = await Runner.run(manager, "Research the benefits of meditation.")
print(result.final_output)

## 6. Conditional Availability with is_enabled

Use `is_enabled` to dynamically control whether an agent-tool is available based on context.

In [ ]:
from agents import RunContextWrapper
from dataclasses import dataclass


@dataclass
class AppContext:
    enable_premium_features: bool = False


def premium_only(ctx: RunContextWrapper[AppContext], agent: Agent) -> bool:
    return ctx.context.enable_premium_features


premium_analyst = Agent(
    name="Premium Analyst",
    instructions="You provide deep analytical insights.",
)

assistant = Agent(
    name="Assistant",
    instructions="You help users. Use the analyst tool if available for deeper analysis.",
    tools=[premium_analyst.as_tool(
        tool_name="deep_analysis",
        tool_description="Perform deep analysis (premium feature)",
        is_enabled=premium_only,
    )],
)

ctx_free = AppContext(enable_premium_features=False)
result = await Runner.run(assistant, "Analyze market trends.", context=ctx_free)
print(f"Free tier: {result.final_output}")

ctx_premium = AppContext(enable_premium_features=True)
result = await Runner.run(assistant, "Analyze market trends.", context=ctx_premium)
print(f"Premium tier: {result.final_output}")

## Key Takeaways

- `agent.as_tool()` turns an agent into a tool that returns its result to the calling agent
- Unlike handoffs, the orchestrator retains control and continues reasoning after the tool call
- The orchestrator pattern lets a manager agent delegate to specialist agents and synthesize results
- `custom_output_extractor` transforms how the specialist's output is returned
- `is_enabled` dynamically controls whether the agent-tool is available based on context